[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/08_soft_distance.ipynb)

# 08 · Soft PPCA lenses, σ² by EMD, and document distance

Hard *k*-means and a single lens map (notebook 03) assign each cluster to one framing. A richer Address layer treats each lens as a **probabilistic PCA** Gaussian over its member centroids and gives every cluster soft mixture weights over lenses. The shared noise scale σ² is not guessed: we sweep it and pick the value that minimises mean **Earth Mover's Distance** between independent lens runs, subject to a meaningful-soft band on effective support (so we do not collapse to the near-uniform solution).

Separately, documents can be compared by EMD on their **cluster share** profiles (cosine ground cost between centroids), with classical MDS and PERMANOVA / PERMDISP for group tests — without using lens names at all.

Requires the optional dependency ``pot`` (`pip install 'multilogue[distance]'`).

In [ ]:
#@title Setup (run first)
# Free path: synthetic units + FakeClient (no API key).
# Optional: load("polis", ...) downloads public open data into a local cache.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from IPython.display import display, HTML, Markdown
from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pot"])
import numpy as np
import pandas as pd
from multilogue import soft as ms
from multilogue import distance as dd
from multilogue import viz

rng = np.random.default_rng(0)
k, dim, n_lenses, n_runs = 24, 12, 6, 3
centroids = rng.normal(size=(k, dim))
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)
mappings = []
for _ in range(n_runs):
    perm = rng.permutation(k)
    mappings.append({f"L{j}": {"cluster_ids": perm[j::n_lenses].tolist()}
                     for j in range(n_lenses)})

sweep = ms.sigma2_sweep_select(centroids, mappings)
curve = pd.DataFrame(sweep["curve"])
print(f"Selected σ²: {sweep['sigma2_best']:.4g} ({sweep['selection']})")
print(f"Mean EMD at σ²*: {sweep['emd_best']:.4f}")
print(f"Mean eff. support: {sweep['eff_support_best']:.2f}")
display(curve[["sigma2", "mean_emd", "mean_eff_support"]].head(8))

W, names = ms.compute_mixture_weights(
    centroids, mappings[0], sweep["sigma2_best"])
print("Mixture weights shape:", W.shape, "lenses:", names)
print("Row sums:", np.round(W.sum(axis=1)[:5], 3))

## Document EMD and PERMANOVA

Plant two groups with different cluster profiles. Documents with too few phrases are dropped; the rest are placed by transport distance and tested for group location differences.

In [ ]:
prof_a = np.array([0.4, 0.4, 0.05, 0.05, 0.05, 0.05])
prof_b = np.array([0.05, 0.05, 0.05, 0.05, 0.4, 0.4])
docs = {f"a{i}": prof_a for i in range(8)}
docs.update({f"b{i}": prof_b for i in range(8)})
rows = []
for doc, profile in docs.items():
    for c in rng.choice(6, size=120, p=profile):
        rows.append({"doc_id": doc, "cluster": int(c)})
phrases = pd.DataFrame(rows)
group = {d: ("A" if d.startswith("a") else "B") for d in docs}
C6 = centroids[:6]
res = dd.run_document_distance_analysis(
    phrases, C6, group, min_phrases=50, n_perm=199, seed=0)
pm = res["permanova"]
print(f"Docs kept: {len(res['docs'])}; dropped: {res['dropped_docs']}")
print(f"PERMANOVA R²={pm['R2']:.3f}, p={pm['p_value']:.4f}")
display(res["pairwise"][["comparison", "R2", "p_value", "dispersion_p"]])

import matplotlib.pyplot as plt
coords = res["mds"]["coords"]
fig, ax = plt.subplots(figsize=(5, 4))
for g, marker in (("A", "o"), ("B", "s")):
    m = res["groups"] == g
    ax.scatter(coords[m, 0], coords[m, 1], label=g, marker=marker)
ax.set_xlabel("MDS 1"); ax.set_ylabel("MDS 2")
ax.set_title("Document EMD → classical MDS")
ax.legend()
fig.tight_layout()
path = FIG / "08_doc_mds.png"
fig.savefig(path, dpi=120)
plt.close(fig)
display(Markdown(f"![MDS]({path})"))